In [4]:

import os
from pyspark.sql import DataFrame, SparkSession

from policy_etl.spark import get_spark



GOLD_TABLES = ["dim_product", "dim_customer", "dim_region", "dim_date", "fact_policy_event"]

import os

for prefix in ("/opt/homebrew", "/usr/local"):
    java_home = f"{prefix}/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"
    if os.path.isdir(java_home):
        os.environ["JAVA_HOME"] = java_home
        os.environ["PATH"] = f"{java_home}/bin:" + os.environ["PATH"]
        break

print(os.environ["JAVA_HOME"])

def register_gold_views(spark: SparkSession, gold_path: str) -> None:
    """Expose each gold table as a SQL view, e.g. fact_policy_event, dim_product."""
    for table in GOLD_TABLES:
        spark.read.parquet(os.path.join(gold_path, table)).createOrReplaceTempView(table)


def run_query(spark: SparkSession, sql_file: str) -> DataFrame:
    with open(sql_file) as f:
        return spark.sql(f.read())


/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home


In [5]:
spark = get_spark(app_name="policy_etl_queries")
spark.sparkContext.setLogLevel("ERROR")
register_gold_views(spark, "data/gold")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/02 23:59:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
spark.sql("SELECT * FROM fact_policy_event LIMIT 5").show()

+--------------------+---------+------------+--------------------+--------------------+--------------------+--------------------+--------+--------------+---------------+--------------+--------------------+--------------------+--------------------+--------------------+----------+
|            event_id|policy_id|  event_type|            event_ts|        customer_key|         product_key|          region_key|date_key|premium_amount|coverage_amount|age_of_insured|            dq_flags|        _source_file|           _batch_id|      _silver_run_id|event_date|
+--------------------+---------+------------+--------------------+--------------------+--------------------+--------------------+--------+--------------+---------------+--------------+--------------------+--------------------+--------------------+--------------------+----------+
|353a24c63c7b7bcf3...|POL-15908|       claim|2024-01-29 06:40:...|-7250998235834192189| 5521022943382472476|-5887901547732339532|20240129|       1786.38|       

In [8]:
spark.sql("SELECT brand, count(*) from fact_policy_event left join dim_product using (product_key) group by brand order by count(*) desc").show()

+-----------+--------+
|      brand|count(1)|
+-----------+--------+
|ProtectPlus|    1304|
| InsureCorp|    1249|
| LifeSecure|    1233|
|  SafeGuard|    1214|
+-----------+--------+



In [9]:
spark.sql("SELECT brand, sum(coverage_amount) from fact_policy_event left join dim_product using (product_key) group by brand order by count(*) desc").show()

+-----------+--------------------+
|      brand|sum(coverage_amount)|
+-----------+--------------------+
|ProtectPlus|         69853588.83|
| InsureCorp|         68312923.48|
| LifeSecure|         67034877.08|
|  SafeGuard|         65647243.47|
+-----------+--------------------+



In [10]:
spark.sql("SELECT AVG(age_of_insured), product_type FROM fact_policy_event left join dim_product using (product_key) group by product_type order by AVG(age_of_insured) desc").show()

+-------------------+------------+
|avg(age_of_insured)|product_type|
+-------------------+------------+
| 45.851866883116884|     unknown|
| 45.632075471698116|      health|
|  45.34248788368336|        life|
|   45.3255033557047|        auto|
|  45.24052718286656|        home|
+-------------------+------------+



In [17]:
spark.sql("""WITH ranked_data AS (SELECT *, DENSE_RANK() OVER (PARTITION BY brand, product_type ORDER BY age_of_insured ASC) AS rank FROM fact_policy_event LEFT JOIN dim_product USING (product_key) where age_of_insured is not null) 
SELECT distinct brand, product_type, age_of_insured FROM ranked_data WHERE rank = 1""").show()

+-----------+------------+--------------+
|      brand|product_type|age_of_insured|
+-----------+------------+--------------+
| InsureCorp|        auto|             1|
| InsureCorp|      health|             1|
| InsureCorp|        home|             0|
| InsureCorp|        life|            17|
| InsureCorp|     unknown|             0|
| LifeSecure|        auto|            17|
| LifeSecure|      health|            17|
| LifeSecure|        home|            17|
| LifeSecure|        life|            17|
| LifeSecure|     unknown|             0|
|ProtectPlus|        auto|            17|
|ProtectPlus|      health|             0|
|ProtectPlus|        home|            17|
|ProtectPlus|        life|            17|
|ProtectPlus|     unknown|             0|
|  SafeGuard|        auto|            18|
|  SafeGuard|      health|             3|
|  SafeGuard|        home|             0|
|  SafeGuard|        life|             0|
|  SafeGuard|     unknown|            17|
+-----------+------------+--------

26/11/03 00:57:11 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:123)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:688)
	at org.apache.spark.storage.BlockManagerMasterE